In [1]:
"""
PROJECT: Software Effort Estimation Using Machine Learning
DATASET: Desharnais Software Project Dataset
OBJECTIVE: Predict software development effort from project features.

This notebook implements a regression-based software effort estimation workflow.
It uses historical software project data to train and evaluate machine learning models.
The dataset contains 77 software project records and 6 numerical columns.
The input features describe project characteristics and team experience.
The target variable is Effort, representing software development effort.
TeamExp represents the development team's experience level.
ManagerExp represents the project manager's experience level.
Transactions represents the number of transactions in the project.
Entities represents the number of entities in the project.
PointsAdjust represents the adjusted function point measure.
Effort represents the recorded development effort.
Pandas is used for loading and manipulating the dataset.
NumPy is used for numerical calculations and target transformation.
Matplotlib is imported for plotting and visualization.
Scikit-learn provides model evaluation and cross-validation utilities.
XGBoost provides a gradient-boosted tree regression model.
Random Forest is used as a second regression model for comparison.
The dataset is loaded from the processed CSV file.
Initial inspection displays a sample of the dataset.
Dataset dimensions are checked to understand its size.
Column data types are inspected for consistency.
Missing values are counted for every column.
Duplicate records are checked during data validation.
The Effort column is verified as the required target variable.
Effort values are converted to a numeric data type.
Missing or negative effort values are identified as invalid.
Descriptive statistics summarize the distribution of effort.
The input matrix X contains all predictor features.
The target vector y contains the original effort values.
A logarithmic transformation is applied to the target.
The log1p transformation helps reduce the influence of large values.
XGBoost is configured for squared-error regression.
The model uses 300 boosting estimators.
Its learning rate is set to 0.05.
Its maximum tree depth is set to 4.
A fixed random seed supports reproducible experiments.
Ten-fold cross-validation evaluates model generalization.
The folds are shuffled using a fixed random seed.
R-squared scores are calculated for each XGBoost fold.
The mean cross-validation R-squared is reported.
A Random Forest regressor is configured for comparison.
The Random Forest model uses 500 decision trees.
Both models use the same cross-validation splitting strategy.
Pearson correlation coefficients are calculated between numeric columns.
Feature correlations provide an initial view of linear relationships.
Correlation does not establish causation.
Cross-validated predictions are generated for each model.
Predictions are initially produced on the logarithmic target scale.
The predictions are converted back to the original effort scale.
The inverse transformation uses the exponential-minus-one operation.
Actual effort values are retained on the original scale.
R-squared measures the proportion of target variance explained.
Mean Absolute Error measures average absolute prediction error.
Root Mean Squared Error penalizes larger prediction errors more strongly.
Evaluation metrics are collected in a results table.
The results table enables direct comparison of the two models.
Higher R-squared generally indicates better predictive performance.
Lower MAE generally indicates smaller absolute prediction errors.
Lower RMSE generally indicates smaller squared-error penalties.
Negative cross-validation R-squared can occur on small test folds.
The dataset is small, so evaluation results may vary across splits.
Results should be interpreted cautiously due to limited sample size.
Feature correlations alone are not a measure of model importance.
No causal conclusions should be drawn from these observational records.
The notebook supports experimentation with historical effort estimation.
Further validation on independent projects is recommended.
Hyperparameter tuning may improve predictive performance.
Alternative validation strategies may also be investigated.
The workflow emphasizes data quality and reproducible evaluation.
The final comparison reports metrics in original effort units.
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/desharnais.arff.csv")
df.head()

,TeamExp,ManagerExp,Transactions,Entities,PointsAdjust,Effort
0,1,4,253,52,305,5152
1,0,0,197,124,321,5635
2,4,4,40,60,100,805
3,0,0,200,119,319,3829
4,0,0,140,94,234,2149


In [3]:
# Data-quality validation for the Desharnais dataset

print("Dataset shape:", df.shape)
print("\nColumn data types:")
display(df.dtypes.rename("Data Type").to_frame())

print("\nMissing values by column:")
display(df.isna().sum().rename("Missing Values").to_frame())

print("\nDuplicate rows:", df.duplicated().sum())

# Validate the target used for software effort estimation
if "Effort" not in df.columns:
    raise ValueError("Required target column 'Effort' is missing.")

df["Effort"] = pd.to_numeric(df["Effort"], errors="coerce")

print("\nEffort target summary:")
display(df["Effort"].describe().to_frame())

invalid_effort = df["Effort"].isna() | (df["Effort"] < 0)
print("Missing or negative Effort values:", int(invalid_effort.sum()))

Dataset shape: (77, 6)

Column data types:


,Data Type
TeamExp,int64
ManagerExp,int64
Transactions,int64
Entities,int64
PointsAdjust,int64
Effort,int64



Missing values by column:


,Missing Values
TeamExp,0
ManagerExp,0
Transactions,0
Entities,0
PointsAdjust,0
Effort,0



Duplicate rows: 0

Effort target summary:


,Effort
count,77.000000
mean,4833.909091
std,4188.185109
min,546.000000
25%,2331.000000
50%,3542.000000
75%,5817.000000
max,23940.000000


Missing or negative Effort values: 0


In [4]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

(77, 6)
<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   TeamExp       77 non-null     int64
 1   ManagerExp    77 non-null     int64
 2   Transactions  77 non-null     int64
 3   Entities      77 non-null     int64
 4   PointsAdjust  77 non-null     int64
 5   Effort        77 non-null     int64
dtypes: int64(6)
memory usage: 3.7 KB
None
TeamExp         0
ManagerExp      0
Transactions    0
Entities        0
PointsAdjust    0
Effort          0
dtype: int64


In [5]:
X = df.drop(columns=["Effort"])
y = df["Effort"]

In [6]:
y_log = np.log1p(y)

In [7]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

In [8]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

r2_scores = cross_val_score(xgb_model, X, y_log, cv=kf, scoring="r2")

print("R2 scores for each fold:", r2_scores)
print("Mean R2:", r2_scores.mean())

R2 scores for each fold: [ 0.19192692  0.37511884 -0.43168482 -0.08475001  0.15987338 -0.41982433
  0.1273156   0.67973314  0.40366261  0.46027015]
Mean R2: 0.14616414995061103


In [9]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    random_state=42
)

rf_scores = cross_val_score(rf_model, X, y_log, cv=kf, scoring="r2")

print("RF R2 scores:", rf_scores)
print("RF Mean R2:", rf_scores.mean())

RF R2 scores: [ 0.3181771   0.39763461 -0.45893976  0.15396456  0.28803657 -0.31685417
  0.16990931  0.40555919  0.5622281   0.61035374]
RF Mean R2: 0.21300692645411132


In [10]:
corr = df.corr()

print(corr["Effort"].sort_values(ascending=False))

Effort          1.000000
PointsAdjust    0.703691
Transactions    0.583127
Entities        0.500223
TeamExp         0.259288
ManagerExp      0.160075
Name: Effort, dtype: float64


In [11]:
# Evaluate cross-validation predictions in original Effort units

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "XGBoost": xgb_model,
    "Random Forest": rf_model
}

results = []

for model_name, model in models.items():
    # Predict the log-transformed target using the same CV strategy
    log_predictions = cross_val_predict(model, X, y_log, cv=kf)

    # Convert predictions back to original Effort units
    predictions = np.expm1(log_predictions)
    actual = y.to_numpy()

    results.append({
        "Model": model_name,
        "R2 (original scale)": r2_score(actual, predictions),
        "MAE (Effort units)": mean_absolute_error(actual, predictions),
        "RMSE (Effort units)": np.sqrt(mean_squared_error(actual, predictions))
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(3))

,R2 (original scale),MAE (Effort units),RMSE (Effort units)
Model,,,
XGBoost,0.266,2339.944,3563.804
Random Forest,0.291,2246.507,3504.131
